# Multi-view Merging as Importance Sampling

Abstract formulation of the multi-view merging problem in the language of self-normalised importance sampling. No reference to any specific application — purely the mathematical structure.


## 1. Setup

Let $X = \mathbb{R}^D$ be the parameter space. We have $V$ probabilistic models — one per **view** — each providing a conditional density over $X$:

$$p_1, p_2, \ldots, p_V \;:\; X \to \mathbb{R}_{\geq 0}.$$

Each $p_v$ represents one source of evidence about an unknown true parameter $x^{\star} \in X$. For each $p_v$ we have:

- **(E) Evaluation oracle**: query $p_v(x)$ at any $x \in X$.
- **(S) Sampling oracle**: draw i.i.d. samples $x \sim p_v$.


## 2. Conditional independence and the fused posterior

The $V$ evidence sources are assumed conditionally independent given $x^{\star}$. Under a flat prior, Bayes' rule gives the **fused posterior**

$$\pi(x) \;:=\; p\!\left(x \mid \mathrm{ev}_1, \ldots, \mathrm{ev}_V\right) \;=\; \frac{1}{Z} \prod_{v=1}^{V} p_v(x)$$

with normalising constant

$$Z = \int_X \prod_{v=1}^{V} p_v(x) \, dx.$$

$Z$ is generally intractable; only the unnormalised target $\tilde{\pi}(x) = \prod_v p_v(x)$ is computable.


## 3. Goal

Estimate functionals of $\pi$ — most commonly the mean:

$$x^{\star} \;\approx\; \mu_\pi \;=\; \mathbb{E}_\pi[X] \;=\; \int x \, \pi(x) \, dx \;=\; \frac{1}{Z} \int x \prod_{v=1}^{V} p_v(x) \, dx.$$


## 4. Why this is an importance-sampling problem

- We cannot sample from $\pi$ directly.
- We only know $\pi$ up to the unknown constant $Z$.
- We have efficient samplers for each $p_v$ (from **S**) and evaluators for each $p_v$ (from **E**).

This is exactly the IS template: an unnormalised target with accessible-but-different proposal distributions.


## 5. The IS estimator

Pick a proposal $q : X \to \mathbb{R}_{\geq 0}$ with $q(x) > 0$ wherever $\pi(x) > 0$. Draw $x_1, \ldots, x_N \sim q$ and form **unnormalised weights**

$$\tilde{w}(x) \;=\; \frac{\tilde{\pi}(x)}{q(x)} \;=\; \frac{\prod_{v=1}^{V} p_v(x)}{q(x)}.$$

The **self-normalised IS** (SNIS) estimate of the mean:

$$\hat{\mu}_\pi \;=\; \frac{\sum_{k=1}^{N} \tilde{w}(x_k) \cdot x_k}{\sum_{k=1}^{N} \tilde{w}(x_k)}.$$

The unknown $Z$ cancels in the ratio. The estimator is **consistent** as $N \to \infty$, and **biased** for finite $N$.


## 6. Choice of proposal

Three natural choices, each making a different trade-off between coverage of $\pi$ and computational simplicity.


### (a) Single-source proposal

Pick one view $i$, set $q = p_i$. Weights simplify:

$$\tilde{w}(x) \;=\; \prod_{v \neq i} p_v(x).$$

Sample from view $i$, reweight by the product of *other* views' densities. Easy to implement; coverage is limited to the typical set of one view.


### (b) Mixture proposal

Use the equal-weight mixture across all views:

$$q_{\text{mix}}(x) \;=\; \frac{1}{V} \sum_{v=1}^{V} p_v(x).$$

Sample from $q_{\text{mix}}$ (e.g. pick a random $i$ then sample $x \sim p_i$), and weight as

$$\tilde{w}(x) \;=\; \frac{\prod_v p_v(x)}{q_{\text{mix}}(x)}.$$

Covers the union of all views' typical sets. Standard balance-heuristic MIS.


### (c) Pool-across-sources (leave-one-out)

For each view $i$, draw $S$ samples $x_i^k \sim p_i$ to get $V \cdot S$ total candidates. Each sample is weighted using its source identity:

$$\tilde{w}(x_i^k) \;=\; \prod_{v \neq i} p_v(x_i^k).$$

Pool all $V \cdot S$ candidates and softmax-normalise. This is *not* the standard balance heuristic — it's closer to a leave-one-out scheme. It avoids cross-source double-counting in the numerator.


## 7. When this works

The estimator behaves well when:

1. The proposal $q$ covers the support of $\pi$.
2. The ratio $\tilde{\pi}/q$ has finite second moment under $q$ — otherwise the CLT fails and infinite-variance pathologies apply.
3. The proposal $q$ is roughly shaped like $\pi$.

**Why merging via product helps.** If the per-view posteriors $p_v$ have uncorrelated modes around $x^{\star}$, the product $\pi = \prod_v p_v$ has its mode at the *intersection* — closer to $x^{\star}$ than any individual $p_v$'s mode. SNIS gives a fused estimate of $x^{\star}$ better than any single-view estimate.

**Gaussian closed form.** For $p_v = \mathcal{N}(\mu_v, \Sigma_v)$, the product is also Gaussian with

$$\Sigma_\pi \;=\; \left(\sum_v \Sigma_v^{-1}\right)^{-1}, \qquad \mu_\pi \;=\; \Sigma_\pi \sum_v \Sigma_v^{-1} \mu_v.$$

Precision-weighted combination — provably tighter covariance than any individual $\Sigma_v$.


## 8. When this fails

The classical IS failure modes apply directly.

### Weight collapse

In high $D$, $\log \tilde{w}$ has variance $\sim \mathcal{O}(D)$. Softmax concentrates on one sample; the **effective sample size**

$$\mathrm{ESS} \;=\; \frac{\left(\sum_k \tilde{w}_k\right)^2}{\sum_k \tilde{w}_k^2}$$

collapses far below $N$.

### Off-distribution evaluation

$\tilde{w}$ requires evaluating $p_v$ at samples drawn from $p_{v' \neq v}$. If $p_v$ is a learned approximation, it may be poorly calibrated outside its own typical set, injecting noise into the weights.

### Mismatched-mode trap

If views' modes are far apart, no sample from $p_i$ lands in $p_j$'s high-density region. All weights tiny; only the most-compromised sample wins.

### Cross-correlation between sources

If conditional independence fails (shared systematic errors across views), the product over-counts evidence and the implied posterior is over-sharp. The IS estimator inherits this over-confidence.


## 9. Summary

We have $V$ density models over a common parameter space and want the mean of their unnormalised product. We can sample from each and evaluate each at any point. This is a self-normalised IS problem with target

$$\pi(x) \;\propto\; \prod_{v=1}^V p_v(x)$$

and proposal any individual $p_v$ or a mixture over them.

The estimator's quality is bounded by:

- How well each $p_v$ is calibrated at off-its-own-typical-set query points.
- How heavily the weight distribution concentrates in high dimension.
- Whether $\pi$'s high-density region is actually covered by the chosen $q$.

The classical IS warnings — weight collapse, infinite variance, support mismatch — apply.
